# fastcharts — demo

Millions of points, interactive, in a notebook. Pan by dragging, zoom with the wheel, double-click to reset.
Zooming a decimated line re-decimates just the visible window (M4), re-centering the f32 offset on the viewport.

In [ ]:
import numpy as np
from fastcharts import Figure
import fastcharts.kernels as k

rng = np.random.default_rng(0)
print("kernel backend:", k.BACKEND)

## 10M-point line (ships decimated: ≤4 points per pixel column)

In [ ]:
n = 10_000_000
t = np.datetime64("2015-01-01") + np.arange(n).astype("timedelta64[s]")
y = np.cumsum(rng.normal(size=n))
y[n // 2] += 80  # a single-sample spike M4 must preserve

fig = Figure(title="10M points — wheel to zoom into the spike", width=950, height=420)
fig.line(t, y, name="random walk")
fig

## Scatter with per-point color + size (Tier 0, GPU picking)
Hover a point: the exact row is read from the f64 canonical store kernel-side (§16).

In [ ]:
m = 20_000
xs = rng.normal(0, 1, m)
ys = xs * 0.5 + rng.normal(0, 0.6, m)
depth = xs**2 + ys**2                  # continuous -> viridis colormap
weight = np.abs(rng.normal(size=m))    # continuous -> size 3..18 px

fig2 = Figure(title="colored + sized scatter", width=950, height=420)
fig2.scatter(xs, ys, color=depth, colormap="viridis", size=weight, size_range=(3, 18), opacity=0.7)
fig2

## Categorical color (factorized -> CVD-safe palette)

In [ ]:
labels = np.array(["alpha", "beta", "gamma"])[rng.integers(0, 3, m)]
fig3 = Figure(title="categorical scatter", width=950, height=420)
fig3.scatter(xs, ys, color=labels, size=4.0, opacity=0.6)
fig3

## 30M-point scatter -> Tier-2 density surface (§5)
Above ~200k points scatter auto-aggregates: the kernel bins the viewport into a
count grid, the client colormaps it on the GPU, and zoom re-bins the visible window.
VRAM stays screen-bounded regardless of point count.

In [ ]:
big = 30_000_000
bx = rng.normal(0, 1, big)
by = bx * 0.7 + rng.normal(0, 0.5, big)

fig4 = Figure(title="30M points (density)", width=950, height=420)
fig4.scatter(bx, by, colormap="magma")
print("tier:", fig4.build_payload()[0]["traces"][0]["tier"])
fig4

## The memory story (§27: if it isn't in the report, it isn't real)

In [ ]:
report = fig.memory_report()
print(f"canonical: {report['canonical_bytes'] / 2**20:.1f} MB kernel-side")
print(f"first-paint transport: {report['transport_bytes_first_paint'] / 2**10:.1f} KB")
print(f"transport bytes/point: {report['transport_bytes_per_point']:.4f}")

## Standalone HTML export (no kernel needed to view)

In [ ]:
fig2.to_html("scatter_chart.html")
print("wrote scatter_chart.html")